# FlightRadar Analytics Project

## 0. Introducción

El objetivo de este proyecto es construir una aplicación de análisis de datos utilizando Apache Spark sobre el dataset que vamos a crear utilizando la API FlightRadar, de la que vamos a sacar información sobre todos los vuelos que han tenido o bien origen o bien destino en los aeropuertos más importantes de España en el último mes.

El dataset va a contener información sobre todos los vuelos que han tenido origen o destino en España en sus aeropuertos más famosos en el último mes. Cada registro representa un vuelo e incluye información sobre aeropuerto de origen, aeropuerto de llegada, aerolínea, información sobre el avión (matrícula, modelo...), día y hora de salida, día y hora de llegada...

La elección de este dataset permite trabajar con un caso de uso muy cercano a un entorno real de Big Data, ya que combina grandes volúmenes de información, datos temporales, datos geográficos, métricas propias del sistema de vuelos. Además, permite aplicar de forma práctica muchas de las funcionalidades principlaes de Apache Spark.

A lo largo del proyecto se seguirá un flujo completo de trabajo, comenzando por la creación del dataset desde datos que vamos a obtener desde la API FlightRadar, análisis del esquema del dataset, la limpieza de datos y las validaciones de calidad. Posteriormente, se enriquecerá el dataset mediante la incorporación de variables derivadas de las ya creadas en el dataset de inicio que nos facilitarán el análisis exploratios, la generación de KPIs y el análisis en tiempo real.

El proyecto también incluirá consultas con DataFrames y Spark SQL, uso de window functions, creación de indicadores de negocio, visualizaciones, simulación de procesamiento en streaming con Structured Streaming.

Con este proyecto se busca no solo analizar vuelos en el últimos mes en los aeropuertos más importantes de España, sino también entender cómo se construye una aplicación Spark completa, desde la ingesta y preparación de los datos hasta la obtención de insights.

## 1. Objetivos

El objetivo principal del proyecto es construir una aplicación completa de análisis de datos con Apache Spark creando nuestro propio dataset utillizando la API FlightRadar sobre en nuestro caso vuelos en el último mes en los aeropuertos más importantes de España.

Los objetivos específicos son:

- Crear nuestro propio DataFrame
- Comprender en profundidad el esquema y significado de las columnas del dataset.
- Limpiar y validar los datos, detectando nulos, duplicados, outliers e inconsistencias.
- Enriquecer el dataset.
- Crear variables derivadas mediante feature engineering.
- Realizar análisis exploratorio con DataFrames y Spark SQL.
- Aplicar Window Functions para análisis avanzados.
- Definir KPIs relevantes del sistema de vuelos en España.
- Construir visualizaciones y dashboards.
- Simular procesamiento en tiempo real con Structured Streaming.
- Documentar limitaciones, aprendizajes y conclusiones del proyecto.

## ¿Qué es un DataFrame?
Un DataFrame es una colección con filas y columnas bien definidas. Cada columna debe tener el mismo número de filas que el resto de columnas y cada columna tiene el mismo tipo de información que debe ser consistente para cada fila en la colección.

## 2. Creación del DataFrame
Hemos optado por crear un dataset sobre los vuelos en el último mes en los que han participado los aeropuertos más importantes de España. Todo esto con la API FlightRadar como base, a través de esta API podemos obtener información de todos los vuelos que ocurren a través de todo el mundo, aunque como hemos comentado ya varias veces nos hemos centrado en los que han ocurrido en el último mes en los aeropuertos más importantes de España.

Lo que hemos hecho en primer lugar es ejecutar el script que se encuentra en la siguiente ruta: **/Volumes/vuelos_españa/default/scripts/carga_vuelos_ultimo_mes.py**
En este script vamos sacando mediante llamadas a la API FlightRadar día a día de los últimos 30 días un archivo .csv por cada día con la información de los vuelos en los que ha intervenido un aeropuerto de los considerados importantes por nosotros en el último mes. Por lo que tras finalizar correctamente la ejecución del script lo que obtenemos es una carpeta con 30 archivos (uno por cada día) .csv con información de los vuelos que se han llevado a cabo cada día.

Mediante la lectura de estos archivos .csv creamos el DataFrame inicial, en el que tenemos la siguiente información: id del vuelo, número del vuelo, indicativo, icao aerolínea, modelo avión, matrícula avión, icao aeropuerto origen, icao aeropuerto destino, fecha hora despegue, fecha hora aterrizaje. Esta información es muy útil pero consideramos que tener solamente información sobre el icao tanto de aeropuertos y aerolíneas es muy insuficiente por lo que ejecutamos otros 2 scripts para obtener más información sobre aerolíneas y aeropuertos.

El ICAO es un código de 4 letras que identifica de forma única a un aeropuerto o aerolínea a nível internacional. Como nosotros no solo queremos tener este código que individualmente es muy poco descriptivo vamos a sacar más información sobre aerolíneas y aeropuertos.

Ejecutamos el siguiente script: **/Volumes/vuelos_españa/default/scripts/Aerolineas.py**
Con este script lo que hacemos es sacar a un archivo .csv el nombre de la aerolínea al que hace referencía el código ICAO, y su IATA (suele tener 3 letras y es lo que suele ver el pasajero en los billetes, buscadores, etc...) y con esto tenemos un archivo .csv con mucha más información sobre las aerolíneas.

Por último ejecutamos el siguiente script: **/Volumes/vuelos_españa/default/scripts/Aeropuertos.py**
Con este script lo que hacemos es sacar a un archivo .csv el nombre del aeropuerto al que hace referencía el código ICAO, y su IATA (suele tener 3 letras y es lo que suele ver el pasajero en los billetes, buscadores, etc...) y con esto tenemos un archivo .csv con mucha más información sobre los aeropuertos.

Creamos un DataFrame con los datos del fichero .csv sobre las aerolíneas y otro DataFrame con los datos del fichero .csv sobre los aeropuertos. Teniendo estos 3 DataFrames hacemos 3 join left (con ICAO aerolínea, ICAO aeropuerto origen e ICAO aeropuerto destino) y obtenemos el DataFrame completo con el que vamos a comenzar a trabajar.

El DataFrame con el que vamos a comenzar a trabajar tiene las siguientes columnas: `id`, `aeropuerto_origen_nombre`, `eropuerto_origen_icao`, `aeropuerto_origen_iata`, `aeropuerto_destino_nombre`, `aeropuerto_destino_icao`, `aeropuerto_destino_iata`, `aerolinea_nombre`, `aerolinea_icao`, `aerolinea_iata`, `fecha_hora_despegue`, `fecha_hora_aterrizaje`, `numero_vuelo`, `matricula_avion`, `modelo_avion`, `indicativo`

Este DataFrame es mucho más completo que el incial, es mucho más descriptivo y tenemos mayor capacidad de análisis.

In [0]:
df_vuelos_españa = spark.read.format("csv") \
                .option("header", "true") \
                .option("inferSchema", "true") \
                .load("/Volumes/vuelos_españa/default/vuelos/")

In [0]:
df_vuelos_españa.count()

In [0]:
nuevos_nombres = [
    "id",
    "numero_vuelo",
    "indicativo",
    "aerolinea_icao",
    "modelo_avion",
    "matricula_avion",
    "aeropuerto_origen_icao",
    "aeropuerto_destino_icao",
    "fecha_hora_despegue",
    "fecha_hora_aterrizaje"
]

df_vuelos_españa = df_vuelos_españa.toDF(*nuevos_nombres)

In [0]:
display(df_vuelos_españa.limit(100))

In [0]:
from pyspark.sql.functions import col
df_aerolineas = df_vuelos_españa.select(
    col("aerolinea_icao")
).dropna().distinct()

display(df_aerolineas)

Esto es el código icao de la compañia

In [0]:
from pyspark.sql.functions import col

df_icaos_unicos = (
    df_vuelos_españa
    .select(col("aeropuerto_origen_icao").alias("icao"))
    .union(
        df_vuelos_españa.select(col("aeropuerto_destino_icao").alias("icao"))
    )
    .dropna()
    .distinct()
    .orderBy("icao")
)

df_icaos_unicos.display()

In [0]:
df_vuelos_españa_aerolineas = spark.read.format("csv") \
                .option("header", "true") \
                .option("inferSchema", "true") \
                .load("/Volumes/vuelos_españa/default/aerolineas/")

In [0]:
display(df_vuelos_españa_aerolineas)

In [0]:
df_vuelos_españa_aeropuertos = spark.read.format("csv") \
                .option("header", "true") \
                .option("inferSchema", "true") \
                .load("/Volumes/vuelos_españa/default/aeropuertos/")

In [0]:
display(df_vuelos_españa_aeropuertos)

In [0]:
df_aerolineas_join = df_vuelos_españa_aerolineas.select(
    col("icao").alias("aerolinea_icao_join"),
    col("iata").alias("aerolinea_iata"),
    col("name").alias("aerolinea_nombre")
)

df_aeropuertos_origen = df_vuelos_españa_aeropuertos.select(
    col("icao").alias("origen_icao_join"),
    col("iata").alias("aeropuerto_origen_iata"),
    col("name").alias("aeropuerto_origen_nombre")
)

df_aeropuertos_destino = df_vuelos_españa_aeropuertos.select(
    col("icao").alias("destino_icao_join"),
    col("iata").alias("aeropuerto_destino_iata"),
    col("name").alias("aeropuerto_destino_nombre")
)

df_vuelos_españa_completo = (
    df_vuelos_españa
    .join(
        df_aerolineas_join,
        col("aerolinea_icao") == col("aerolinea_icao_join"),
        "left"
    )
    .join(
        df_aeropuertos_origen,
        col("aeropuerto_origen_icao") == col("origen_icao_join"),
        "left"
    )
    .join(
        df_aeropuertos_destino,
        col("aeropuerto_destino_icao") == col("destino_icao_join"),
        "left"
    )
    .drop(
        "aerolinea_icao_join",
        "origen_icao_join",
        "destino_icao_join"
    )
)

df_vuelos_españa_completo.display()

In [0]:
df_vuelos_españa_completo = df_vuelos_españa_completo.select(
    "id",
    "aeropuerto_origen_nombre",
    "aeropuerto_origen_icao",
    "aeropuerto_origen_iata",
    "aeropuerto_destino_nombre",
    "aeropuerto_destino_icao",
    "aeropuerto_destino_iata",
    "aerolinea_nombre",
    "aerolinea_icao",
    "aerolinea_iata",
    "fecha_hora_despegue",
    "fecha_hora_aterrizaje",
    "numero_vuelo",
    "matricula_avion",
    "modelo_avion",
    "indicativo"
)

df_vuelos_españa_completo.display()

Por último, para finalizar la cración del DataFrame como tenemos los valores de las fechas y hora de despegue y aterrizaje en horario universal lo actualizamos y lo ponemos en formato de hora española.

In [0]:
from pyspark.sql.functions import from_utc_timestamp

df_vuelos_españa_completo = df_vuelos_españa_completo.withColumn(
    "fecha_hora_despegue",
    from_utc_timestamp("fecha_hora_despegue", "Europe/Madrid")
)

In [0]:
df_vuelos_españa_completo = df_vuelos_españa_completo.withColumn(
    "fecha_hora_aterrizaje",
    from_utc_timestamp("fecha_hora_aterrizaje", "Europe/Madrid")
)

In [0]:
df_vuelos_españa_completo.display()

## 3. Limpieza, validaciones y Data Quality
En esta sección se realiza un análisis de calidad del dataset antes de construir las capas analíticas posteriores. El objetivo es identificar problemas como valores nulos, duplicados, outliers, e inconsistencias.
La calidad del dato es una fase crítica del proyecto, ya que cualquier error no detectado en este punto puede afectar a las consultas exploratorias, KPIs y visualizaciones.
Durante esta fase se analizarán los datos originales, se crearán reglas de validación y se construirá un dataset limpio final que servirá como base para el resto del proyecto.

En primer lugar analizaremos el esquema del DataFrame.
## ¿Qué es un Esquema?
Un esquema define los nombres de columnas y tipos de las mismas de un DataFrame. Podemos definir manualmente el esquema o leerlo de una fuente de datos como hemos hecho en esta ocasión.

In [0]:
len(df_vuelos_españa_completo.columns)

In [0]:
df_vuelos_españa_completo.columns

In [0]:
df_vuelos_españa_completo.printSchema()

### Conclusiones del análisis de schema

El dataset final con el que vamos a trabajar contiene 16 columnas con información sobre los vuelos que han tenido lugar en el último mes en los aeropuertos más importantes de España con información sobre el aeropuerto de origen, aeropuerto de destino, aerolínea, módelo de avión, matrícula, hora de despegue y hora de aterrizaje.

El esquema está correctamente estructurado para trabajar con Spark, ya que las columnas descriptivas están tipadas como string `aeropuerto_origen_nombre`, `aeropuerto_origen_icao`, `aeropuerto_origen_iata`, `aeropuerto_destino_nombre`, `aeropuerto_destino_icao`, `aeropuerto_destino_iata`, `aerolinea_nombre`, `aerolinea_icao`, `aerolinea_iata`, `numero_vuelo`, `matricula_avion`, `modelo_avion`, `indicativo`, las columnas con información con fecha y hora están tipadas como timestamp `fecha_hora_despegue`, `fecha_hora_aterrizaje`.

Todas las columnas aparecen como nullable en el schema. Esto no implica necesariamente que todas contengan valores nulos, sino que el esquema permite su existencia por lo que en el siguiente apartado se analizará la presencia real de valores nulos.

En general, el schema es adecuado para continuar con el proceso de validación, limpieza, enriquecimiento y análisis del dataset.

In [0]:
df_vuelos_españa_completo.count()

## Nulls

En esta sección se analiza la presencia de valores nulos en el dataset con el que vamos a trabajar. Aunque el esquema indica que todas las columnas permiten valores nulos, es necesario comprobar cuántos valores faltantes existen realmente y qué impacto pueden tener sobre el análisis posterior.

Los valores nulos pueden afectar a operaciones como agregaciones, cálculos, validaciones, KPIs y visualizaciones. Por ello, antes de aplicar cualquier transformación, se medirá la cantidad de nulos por columna.

In [0]:
from pyspark.sql.functions import isnan, count, when, col
nulls_count_df = df_vuelos_españa_completo.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_vuelos_españa_completo.columns
])
display(nulls_count_df)

In [0]:
from functools import reduce
from pyspark.sql.functions import col

condicion_nulos = reduce(
    lambda acc, c: acc | col(c).isNull(),
    df_vuelos_españa_completo.columns[1:],
    col(df_vuelos_españa_completo.columns[0]).isNull()
)

df_registros_con_nulos = df_vuelos_españa_completo.filter(condicion_nulos)

display(df_registros_con_nulos)

In [0]:
df_vuelos_españa_completo = df_vuelos_españa_completo.dropna(
    subset=["aeropuerto_origen_nombre",
    "aeropuerto_destino_nombre"]
)

In [0]:
condicion_nulos = reduce(
    lambda acc, c: acc | col(c).isNull(),
    df_vuelos_españa_completo.columns[1:],
    col(df_vuelos_españa_completo.columns[0]).isNull()
)

df_registros_con_nulos = df_vuelos_españa_completo.filter(condicion_nulos)

display(df_registros_con_nulos)

In [0]:
nulls_count_df = df_vuelos_españa_completo.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_vuelos_españa_completo.columns
])
display(nulls_count_df)

In [0]:
df_vuelos_españa_completo = df_vuelos_españa_completo\
    .withColumn(
        "sin_aerolínea",
        when(
            df_vuelos_españa_completo.aerolinea_nombre.isNull(),
            True
        ).otherwise(False) 
    )

In [0]:
df_vuelos_españa_completo = df_vuelos_españa_completo\
    .withColumn(
        "sin_fecha_hora_despegue",
        when(
            df_vuelos_españa_completo.fecha_hora_despegue.isNull(),
            True
        ).otherwise(False) 
    )

In [0]:
df_vuelos_españa_completo = df_vuelos_españa_completo\
    .withColumn(
        "sin_fecha_hora_aterrizaje",
        when(
            df_vuelos_españa_completo.fecha_hora_aterrizaje.isNull(),
            True
        ).otherwise(False) 
    )

In [0]:
df_vuelos_españa_completo.display()

Una vez analizado el número de nulos que tenemos y en las columnas que los tenemos hemos tomado las siguientes decisiones.

1. Hemos eliminado los registros con valores nulos o bien en `aeropuerto_origen_nombre` o en `aeropuerto_origen_destino` ya que son las columnas principales y de las que tener información en el resto de columnas cobra sentido. Esto ha implicado eliminar 49 registros, algo realmente insignificante, el 0,01% de los registros.
2. Hemos marcado los valores nulos en `aerolínea_nombre`, `fecha_hora_despegue`, `fecha_hora_aterrizaje` mediante las columnas `sin_aerolínea`, `sin_fecha_despegue`, `sin_fecha_aterrizaje`. Hemos decidido marcarlos mediante estás nuevas columnas y no eliminarlos directamente porque son registros que a pesar de tener valores nulos en algunas de esas columnas aportan valor mediante el resto de columnas del registro y podemos sacar conclusiones a través del resto de columnas.
3. Para el resto de columnas con nulos la decisión ha sido mantenerlo tal cuál ya que son columnas que nos aportan un valor descriptivo adicional sobre el registro pero no son principales ni decisivas en el análisis principal del dataset.

Tomadas estás decisiones podemos concluir que el Dataset es bastante limpio sin realmente muchos valores nulos, y podemos trabajar con él.


## Duplicados

En esta sección se analiza la presencia de registros duplicados en el dataset con el que vamos a trabajar.

Los registros duplicados pueden afectar a operaciones como agregaciones, cálculos, validaciones, KPIs y visualizaciones. Por ello, antes de aplicar cualquier transformación, trataremos de eliminar los registros duplicados.

In [0]:
df_vuelos_españa_completo.dropDuplicates().count()

In [0]:
numero_registros_totales = df_vuelos_españa_completo.count()
numero_registros_unicos = df_vuelos_españa_completo.dropDuplicates().count()

registros_duplicados = numero_registros_totales - numero_registros_unicos
porcentaje_duplicados = (
    registros_duplicados / numero_registros_totales
) * 100

print(f"Registros totales: {numero_registros_totales}")
print(f"Registros unicos: {numero_registros_unicos}")
print(f"Registros duplicados: {registros_duplicados}")
print(f"Porcentaje exacto de duplicados: {porcentaje_duplicados:.4f}%")

Como podemos ver no tenemos registros duplicados, esto es una gran noticia y en gran parte se debe a que ya hemos ido revisando cuándo hemos sacado la información de los vuelos que no tuviesemos datos duplicados por lo que en gran parte era esperado que no tuvieramos registros duplicados en este punto.

## Validaciones

En esta sección se valida la coherencia del dataset. Las columnas `aeropuerto_origen_nombre` y `aeropuerto_destino_nombre`, `fecha_hora_despegue`, `fecha_hora_aterrizaje` son fundamentales para el análisis, ya que nos permiten realizar muchos cálculos y análisis sobre los vuelos que han tenido lugar en España en los aeropuertos más importantes en el último mes.

En este apartado se revisará con detalle que todos los registros tengan o bien como origen o bien como destino un aeropuerto de los más importantes de España, que el aeropuerto de despegue sea distinto del aeropuerto de llegada, que la fecha de aterrizaje sea inferior a la fecha de despegue o que la diferencia entre fecha de despegue y fecha de aterrizaje sea coherente.

In [0]:
aeropuertos_españa = ["LEBL", "LEBB", "LEIB", "GCLA", "LEMD", "LEPA", "LEZL", "LEVC"]

In [0]:
df_vuelos_españa_completo.filter(col("aeropuerto_origen_icao").isin(aeropuertos_españa) | col("aeropuerto_destino_icao").isin(aeropuertos_españa)).count()

In [0]:
df_vuelos_españa_completo.filter(col("aeropuerto_destino_nombre") == col("aeropuerto_origen_nombre")).count()

In [0]:
display(df_vuelos_españa_completo.filter(col("aeropuerto_destino_nombre") == col("aeropuerto_origen_nombre")))

In [0]:
df_vuelos_españa_completo = df_vuelos_españa_completo\
    .withColumn(
        "mismo_origen_destino",
        when(
            col("aeropuerto_origen_nombre") == col("aeropuerto_destino_nombre"),
            True
        ).otherwise(False)
    )

No es normal que el aeropuerto de origen sea el mismo que el aeropuerto de destino pero no vamos a eliminar estos registros, los vamos a marcar ya que a pesar de ser sospechosos es posible que sea real y el vuelo haya tenido que volver al aeropuerto de origen.

In [0]:
df_vuelos_españa.filter(col("fecha_hora_aterrizaje") < col("fecha_hora_despegue")).count()

In [0]:
from pyspark.sql.functions import col, unix_timestamp, abs

df_vuelos_españa_completo\
    .filter(
        abs(unix_timestamp(col("fecha_hora_aterrizaje")) - unix_timestamp(col("fecha_hora_despegue"))) >=  16 * 60 * 60
    ).count()

## Dataset limpio final

En esta sección se consolida el resultado de todas las validaciones realizadas en el bloque de Data Quality.

El objetivo ha sido construir un dataset limpio, trazable y preparado para los siguientes pasos del proyecto: enriquecimiento, feature engineering, análisis exploratorio, KPIs y visualizaciones.

No se han eliminado los registros sospechosos de forma agresiva, excepto, los registros sin aeropuerto de origen o aeropuerto de destino, se han analizado individualmente y se ha puesto una solución para que sean registros útiles y válidos para trabajar con el dataset.

## 4. Creación de variables adicionales
En esta sección creamos una serie de variables adicionales basadas en columnas e información que ya tenemos sobre cada registro para tener un dataset mucho más completo. Metemos columnas como si es un viaje internacional, es un viaje nacional, es un viaje largo, medio o corto, despega en un día y llega en otro, están Barcelona o Madrid involucrados, el aeropuerto de origen es español, el aeropuerto de destino es español
Todas estas variables nos facilitarán todos los cálculos y análisis que hagamos posteriormente.

In [0]:
aeropuertos_españoles_todos = ["LECO", "LEAB", "LEAG", "LEMU", "LEAL", "LEAM", "LESU", "LEAS", "LEAX", "LEBZ", "LEBL", "LEBB", "LEBG", "LECH", "GECE", "LERL", "LEBA", "GCHI", "GCFV", "LEGE", "GCLP", "LEGR", "LEHC", "LEIB", "LEJR", "LEDC", "GCGM", "GCLA", "LEIZ", "GCRR", "LELN", "LEDA", "LERJ", "LECT", "LECU", "LETO", "LEGT", "LEMD", "LEMH", "LEMG", "GCLB", "GEML", "LEMO", "LEMI", "LECL", "LEPA", "LESB", "LEPP", "LERS", "LERT", "LELL", "LESA", "LESO", "LEXJ", "LEST", "LEZL", "GCXO", "GCTS", "LETL", "LEVC", "LEVD", "LEVX", "LEVT", "LEZG"]

In [0]:
df_vuelos_españa_completo = df_vuelos_españa_completo\
    .withColumn(
        "viaje_internacional",
        when(
            col("aeropuerto_origen_icao").isin(aeropuertos_españoles_todos) & col("aeropuerto_destino_icao").isin(aeropuertos_españoles_todos),
            False
        ).otherwise(True)
    )

In [0]:
df_vuelos_españa_completo = df_vuelos_españa_completo\
    .withColumn(
        "origen_español",
        when(
            col("aeropuerto_origen_icao").isin(aeropuertos_españoles_todos),
            True
        ).otherwise(False)
    )\
    .withColumn(
        "destino_español",
        when(
            col("aeropuerto_destino_icao").isin(aeropuertos_españoles_todos),
            True
        ).otherwise(False)
    )

In [0]:
from pyspark.sql.functions import round
df_vuelos_españa_completo = df_vuelos_españa_completo \
    .withColumn(
        "duracion_horas",
        round(
            (
                unix_timestamp(col("fecha_hora_aterrizaje")) -
                unix_timestamp(col("fecha_hora_despegue"))
            ) / 3600,
            2
        )
    )

In [0]:
from pyspark.sql.functions import col, when, to_date

df_vuelos_españa_completo = df_vuelos_españa_completo.withColumn(
    "viaje_nocturno",
    when(
        to_date(col("fecha_hora_despegue")) != to_date(col("fecha_hora_aterrizaje")),
        True
    ).otherwise(False)
)

In [0]:
df_vuelos_españa_completo = df_vuelos_españa_completo\
    .withColumn(
        "viaje_corto",
        when(
            col("duracion_horas") <= 2,
            True
        ).otherwise(False)
    )\
    .withColumn(
        "viaje_mediano",
        when(
            (col("duracion_horas") > 2) & (col("duracion_horas") <= 6),
            True
        ).otherwise(False)
    )\
    .withColumn(
        "viaje_largo",
        when(
            col("duracion_horas") > 6,
            True
        ).otherwise(False)
    )

In [0]:
df_vuelos_españa_completo = df_vuelos_españa_completo\
    .withColumn(
        "madrid_or_barcelona",
        when(
            ((col("aeropuerto_origen_icao") == "LEMD") | (col("aeropuerto_origen_icao") == "LEBL")) | ((col("aeropuerto_destino_icao") == "LEMD") | (col("aeropuerto_destino_icao") == "LEBL")),
            True
        ).otherwise(False)
    )

Una vez creadas todas estas variables adicionales desplegamos el DataFrame para comprobar que se han creado correctamente y que nos aportan la información que buscabamos.
Tras esto ya podemos comenzar con los diferentes cálculos y análisis sobre el DataFrame.

In [0]:
display(df_vuelos_españa_completo)

## 5. Análisis Exploratorio, KPIs y Visualizaciones

En primer lugar vamos a sacar una serie de estadísticas más generales, como % porcentaje de viajes cortos/medios/largos, número de viajes internacionales, número de viajes nacionales, número de viajes que cruzan de día, % de viajes en los que están involucrados Madrid o Barcelona, número de aeropuertos involucrados, número de aerolíneas involucradas, media de duración de los viajes, es decir, un análisis más general.

Más adelante nos centraremos en análisis más específicos sobre aeropuertos, aerolíneas, módelos de avión para acabar analizando estadísticas más simbolicas que de valor realmente.

In [0]:
df_vuelos_españa_completo.filter(~col("sin_fecha_hora_despegue") & (~col("sin_fecha_hora_aterrizaje"))).count()

La mayoría de registros viene con información completa sobre la hora de despegue y la hora de aterrizaje. Esto es muy positivo ya que podremos analizar adecuadamente todo lo relacionado con la duración de los vuelos.

In [0]:
df_vuelos_españa_completo.filter(~col("sin_fecha_hora_despegue") & (~col("sin_fecha_hora_aterrizaje")))\
    .select(
        "aeropuerto_origen_nombre",
        "aeropuerto_destino_nombre",
        "aerolinea_nombre",
        "duracion_horas"
    )\
    .orderBy(col("duracion_horas").desc()).display()

In [0]:
df_vuelos_españa_completo.filter(~col("sin_fecha_hora_despegue") & (~col("sin_fecha_hora_aterrizaje")))\
    .select(
        "aeropuerto_origen_nombre",
        "aeropuerto_destino_nombre",
        "aerolinea_nombre",
        "duracion_horas"
    )\
    .orderBy(col("duracion_horas").asc()).display()

In [0]:
from pyspark.sql.functions import avg
from pyspark.sql.functions import round as spark_round
df_vuelos_españa_completo.filter(~col("sin_fecha_hora_despegue") & (~col("sin_fecha_hora_aterrizaje")))\
    .agg(
        spark_round(
            avg(col("duracion_horas")),
            2).alias("duracion_media_vuelos")
        ).display()

Como podemos observar hay mucha variedad, desde vuelos de media hora entre ciudades de la misma comunidad autónoma hasta vuelos de más de 14 horas a Asia. Pero la duración media de los vuelos nos permite sacar una conclusión, lo que predomina son vuelos a ciudades europeas cuya duración está entre 1,5 horas y 4 horas, de ahí esa duración media.

In [0]:
import builtins
from pyspark.sql.functions import col

numero_vuelos_con_datos_tiempo = df_vuelos_españa_completo.filter(
    (~col("sin_fecha_hora_despegue")) &
    (~col("sin_fecha_hora_aterrizaje"))
).count()

numero_vuelos_cortos = df_vuelos_españa_completo.filter(
    (~col("sin_fecha_hora_despegue")) &
    (~col("sin_fecha_hora_aterrizaje")) &
    (col("viaje_corto"))
).count()

porcentaje_vuelos_cortos = builtins.round(
    numero_vuelos_cortos / numero_vuelos_con_datos_tiempo * 100,
    2
)

print(f"El porcentaje de vuelos cortos es {porcentaje_vuelos_cortos}%")

In [0]:
numero_vuelos_con_datos_tiempo = df_vuelos_españa_completo.filter(
    (~col("sin_fecha_hora_despegue")) &
    (~col("sin_fecha_hora_aterrizaje"))
).count()

numero_vuelos_medios = df_vuelos_españa_completo.filter(
    (~col("sin_fecha_hora_despegue")) &
    (~col("sin_fecha_hora_aterrizaje")) &
    (col("viaje_mediano"))
).count()

porcentaje_vuelos_medios = builtins.round(
    numero_vuelos_medios / numero_vuelos_con_datos_tiempo * 100,
    2
)

print(f"El porcentaje de vuelos medios es {porcentaje_vuelos_medios}%")

In [0]:
numero_vuelos_con_datos_tiempo = df_vuelos_españa_completo.filter(
    (~col("sin_fecha_hora_despegue")) &
    (~col("sin_fecha_hora_aterrizaje"))
).count()

numero_vuelos_largos = df_vuelos_españa_completo.filter(
    (~col("sin_fecha_hora_despegue")) &
    (~col("sin_fecha_hora_aterrizaje")) &
    (col("viaje_largo"))
).count()

porcentaje_vuelos_largos = builtins.round(
    numero_vuelos_largos / numero_vuelos_con_datos_tiempo * 100,
    2
)

print(f"El porcentaje de vuelos largos es {porcentaje_vuelos_largos}%")

Analizados los resultados podemos concluir que predominan los vuelos cortos (inferiores a 2 horas) con más de un 50% entre los que se encuentras muchos vuelos nacionales, incluso dentro de comunidades autónomas y vuelos europeos con duración inferior a las 2 horas. Le siguen los vuelos medios con un 27% (entre 2-6h) entre los que se encuentran muchos vuelos europeos y por último los vuelos de más de 6 horas (largos) que son más anomalos y son entorno al 16% del volúmen de aviones.

In [0]:
df_vuelos_españa_completo.filter(col("viaje_internacional")).count()

In [0]:
df_vuelos_españa_completo.filter(~col("viaje_internacional")).count()

Realmente destacan los vuelos internacionales con casi el doble de vuelos que los nacionales. Esto realmente tiene sentido por lo que venimos comentando se realizan muchos vuelos europeos y si sumamos a eso vuelos a otros contienentes a que no sean tan habituales pues realmente predominan mucho los vuelos internacionales sobre los vuelos nacionales.

In [0]:
numero_vuelos_cortos_internacionales = df_vuelos_españa_completo.filter(
    (~col("sin_fecha_hora_despegue")) &
    (~col("sin_fecha_hora_aterrizaje")) &
    (col("viaje_internacional")) &
    (col("viaje_corto"))
).count()

numero_vuelos_cortos = df_vuelos_españa_completo.filter(
    (~col("sin_fecha_hora_despegue")) &
    (~col("sin_fecha_hora_aterrizaje")) &
    (col("viaje_corto"))
).count()

porcentaje_vuelos_cortos = builtins.round(
    numero_vuelos_cortos_internacionales / numero_vuelos_cortos * 100,
    2
)

print(f"El porcentaje de vuelos internaciones dentro de los cortos es: {porcentaje_vuelos_cortos}%")

Es el tipo de viaje en el que menos predomina el viaje internacional y a pesar de eso más de un 40% son viajes internacionales, como hemos comentado en más de una ocasión ya esto se debe al gran volumen de viajes que hay a otras ciudades europeas a las que el viaje no llega ni a 2 horas.

In [0]:
numero_vuelos_medios_internacionales = df_vuelos_españa_completo.filter(
    (~col("sin_fecha_hora_despegue")) &
    (~col("sin_fecha_hora_aterrizaje")) &
    (col("viaje_internacional")) &
    (col("viaje_mediano"))
).count()

numero_vuelos_medios = df_vuelos_españa_completo.filter(
    (~col("sin_fecha_hora_despegue")) &
    (~col("sin_fecha_hora_aterrizaje")) &
    (col("viaje_mediano"))
).count()

porcentaje_vuelos_medios = builtins.round(
    numero_vuelos_medios_internacionales / numero_vuelos_medios * 100,
    2
)

print(f"El porcentaje de vuelos internaciones dentro de los medios es: {porcentaje_vuelos_medios}%")

Aquí ya si dominan totalmente los viajes internacionales con más del 90%, existen pocos viajes nacionales que duren más de 2 horas.

In [0]:
numero_vuelos_largos_internacionales = df_vuelos_españa_completo.filter(
    (~col("sin_fecha_hora_despegue")) &
    (~col("sin_fecha_hora_aterrizaje")) &
    (col("viaje_internacional")) &
    (col("viaje_largo"))
).count()

numero_vuelos_largos = df_vuelos_españa_completo.filter(
    (~col("sin_fecha_hora_despegue")) &
    (~col("sin_fecha_hora_aterrizaje")) &
    (col("viaje_largo"))
).count()

porcentaje_vuelos_largos = builtins.round(
    numero_vuelos_largos_internacionales / numero_vuelos_largos * 100,
    2
)

print(f"El porcentaje de vuelos internaciones dentro de los largos es: {porcentaje_vuelos_largos}%")

In [0]:
display(df_vuelos_españa_completo.filter(
    (~col("sin_fecha_hora_despegue")) &
    (~col("sin_fecha_hora_aterrizaje")) &
    (~col("viaje_internacional")) &
    (col("viaje_largo"))
))

Sorprende no obtener el 100%, y analizado el único viaje que no cumple que sea internacional es posible que tuviese algún problema a la hora de aterrizar o simplemente sea un dato erroneo, es un registro realmente sospechoso.

In [0]:
df_vuelos_españa_completo.filter((~col("sin_fecha_hora_aterrizaje") & (~col("sin_fecha_hora_despegue")) & col("viaje_nocturno"))).count()

In [0]:
df_vuelos_españa_completo.filter(col("viaje_nocturno")).count()

Comprobar si está bien creada la variable derivada "viaje_nocturno"

In [0]:
df_vuelos_españa_completo.filter((col("aeropuerto_origen_icao") == "LEMD") | (col("aeropuerto_destino_icao") == "LEMD")).count()

In [0]:
numero_vuelos_totales = df_vuelos_españa_completo.count()
numero_vuelos_madrid = df_vuelos_españa_completo.filter((col("aeropuerto_origen_icao") == "LEMD") | (col("aeropuerto_destino_icao") == "LEMD")).count()

porcentaje_madrid = builtins.round(
    numero_vuelos_madrid / numero_vuelos_totales * 100,
    2
)

print(f"El porcentaje de vuelos que tienen como origen o destino Madrid es: {porcentaje_madrid}%")

En casi un 20% de los viajes está involucrado el seropuerto de Madrid, un porcentaje realmente muy alto pero explicable desde el punto de que Madrid es la capital de España y su volumen de viajes es muy alto.

In [0]:
df_vuelos_españa_completo.filter((col("aeropuerto_origen_icao") == "LEBL") | (col("aeropuerto_destino_icao") == "LEBL")).count()

In [0]:
numero_vuelos_totales = df_vuelos_españa_completo.count()
numero_vuelos_barcelona = df_vuelos_españa_completo.filter((col("aeropuerto_origen_icao") == "LEBL") | (col("aeropuerto_destino_icao") == "LEBL")).count()

porcentaje_barcelona = builtins.round(
    numero_vuelos_barcelona / numero_vuelos_totales * 100,
    2
)

print(f"El porcentaje de vuelos que tienen como origen o destino Barcelona es: {porcentaje_barcelona}%")

Es ligeramente inferior el % que tiene el aeropuerto de Barcelona pero con muy poca diferencia realmente, es un aeropuerto con mucho tráfico también.

In [0]:
df_vuelos_españa_completo.filter(col("madrid_or_barcelona")).count()

In [0]:
numero_vuelos_totales = df_vuelos_españa_completo.count()
numero_vuelos_barcelona_madrid = df_vuelos_españa_completo.filter(col("madrid_or_barcelona")).count()

porcentaje_barcelona_madrid = builtins.round(
    numero_vuelos_barcelona_madrid / numero_vuelos_totales * 100,
    2
)

print(f"El porcentaje de vuelos que tienen como origen o destino Barcelona o Madrid es: {porcentaje_barcelona_madrid}%")

Un tercio de los vuelos involucran o bien a Barcelona o bien a Madrid.

In [0]:
df_aeropuertos_origen = df_vuelos_españa_completo\
    .select(
        col("aeropuerto_origen_nombre").alias("aeropuerto_nombre"),
        col("aeropuerto_origen_icao").alias("aeropuerto_icao")
    )

df_aeropuertos_destino = df_vuelos_españa_completo\
    .select(
        col("aeropuerto_destino_nombre").alias("aeropuerto_nombre"),
        col("aeropuerto_destino_icao").alias("aeropuerto_icao")
    )

df_aeropuertos = df_aeropuertos_origen.union(df_aeropuertos_destino)\
    .groupBy(col("aeropuerto_nombre"), col("aeropuerto_icao")).count().orderBy(col("count").desc()).display()

Obviamente los 8 primeros son los aeropuertos de los que hemos sacado todos sus viajes en el último mes, con especial mención a Madrid y Barcelona de los que ya hemos hablado previamente y destacando al Aeropuerto de Palma de Mallorca aunque no sea ninguna sorpresa ya que por la época del año en la que nos encontramos Mallorca como ciudad es una de las más visitadas. 

Hay que destacar el número de viajes en los que participa el Aeropuerto de Tenerife Norte y el de Gran Canaria ya que sin haber sacado sus viajes explicitamente aparecen muy cercanos a otros aeropuertos de los que si hemos sacado todos sus viajes.

Por último destacar el aeropuerto de Frankfurt aunque no es ninguna sorpresa tampoco como comprobaremos luego por su gran conexión con el aeropuerto de Palma de mallorca.

In [0]:
df_vuelos_españa_completo.groupBy(col("aerolinea_icao"), col("aerolinea_nombre")).count().orderBy(col("count").desc()).display()

La aerolínea más utilizada es Vueling además con más de 100 viajes de diferencia con Rayanair. La siguen las aerolíneas Canair e Iberia muy consolidadas. Más adelante veremos si tienen alguna ruta especifica en exclusiva que las hace destacar.

In [0]:
df_vuelos_españa_completo.groupBy(col("modelo_avion")).count().orderBy(col("count").desc()).display()

El módelo de avión más utilizado con mucha diferencia además es el A320 fabricado por airbus, no es ninguna sorpresa ya que es uno de los módelos más vendidos de la historia y está especializado en vuelos de corta o media distancia. Le siguen los módelos B738 y AT76 que también son módelos que se utilizan para corta o media distancia.

## Análisis estadísticas relaciondas con los aeropuertos

Vamos a analizar mas detalladamente todo lo relacionado con los aeropuertos.

Comenzamos analizando de que aeropuerto nacen más viajes y que aeropuerto recibe más viajes

In [0]:
df_vuelos_españa_completo.groupBy(col("aeropuerto_origen_nombre"), col("aeropuerto_origen_icao")).count().orderBy(col("count").desc()).display()

In [0]:
df_vuelos_españa_completo.groupBy(col("aeropuerto_destino_nombre"), col("aeropuerto_destino_icao")).count().orderBy(col("count").desc()).display()

Es muy interesante está estadística, podemos observar como Valencia o Bilbao son los top en cuánto a viajes que nacen en sus aeropuertos pero no destacan en recibir viajes, en ese caso destacan Barcelona, Madrid y Mallorca que reciben muchísimos más viajes de los que originan. De esto podemos sacar la conclusión de que las ciudades más demandadas sin duda son Barcelona, Madrid y Mallorca, tambíen Ibiza en la que ocurre lo mismo. Son ciudades, como respaldan los datos con mucha atracción turística.

Cabe destacar como aparecen en los primeros lugares de aeropuertos de origen los aeropuertos alemanes de Colonia y Frankfurt algo que explicaremos más adelante con las rutas más repetidas y la gran conexión de los aeropuertos alemanes con Mallorca.

In [0]:
df_vuelos_españa_completo.filter(~col("aeropuerto_origen_icao").isin(aeropuertos_españa)).groupBy(col("aeropuerto_origen_nombre"), col("aeropuerto_origen_icao")).count().orderBy(col("count").desc()).display()

In [0]:
df_vuelos_españa_completo.filter(~col("aeropuerto_destino_icao").isin(aeropuertos_españa)).groupBy(col("aeropuerto_destino_nombre"), col("aeropuerto_destino_icao")).count().orderBy(col("count").desc()).display()

In [0]:
df_aeropuertos_origen = df_vuelos_españa_completo.filter(~col("aeropuerto_origen_icao").isin(aeropuertos_españa))\
    .select(
        col("aeropuerto_origen_nombre").alias("aeropuerto_nombre"),
        col("aeropuerto_origen_icao").alias("aeropuerto_icao")
    )

df_aeropuertos_destino = df_vuelos_españa_completo.filter(~col("aeropuerto_destino_icao").isin(aeropuertos_españa))\
    .select(
        col("aeropuerto_destino_nombre").alias("aeropuerto_nombre"), 
        col("aeropuerto_destino_icao").alias("aeropuerto_icao")
    )

df_aeropuertos = df_aeropuertos_origen.union(df_aeropuertos_destino)\
    .groupBy(col("aeropuerto_nombre"), col("aeropuerto_icao")).count().orderBy(col("count").desc()).display()

Cabe destacar de nuevo la gran cantidad de vuelos en los que está involucrado el aeropuerto de Tenerife Norte, más de 400 viajes en los que aparece sin haber sacado todos sus viajes. 

Muy destacable también el aeropuerto de Gran Canaria así como lo que hemos comentado con la conexión europea sobre todo con los alemanes y con Amsterdam y Milan que siempre aparecen en los rankings de mayores apariciones.

In [0]:
from pyspark.sql.functions import col, lit

df_aeropuertos_origen = df_vuelos_españa_completo \
    .filter(~col("aeropuerto_origen_icao").isin(aeropuertos_españoles_todos)) \
    .select(
        col("aeropuerto_origen_nombre").alias("aeropuerto_nombre"),
        col("aeropuerto_origen_icao").alias("aeropuerto_icao")
    )

df_aeropuertos_destino = df_vuelos_españa_completo \
    .filter(~col("aeropuerto_destino_icao").isin(aeropuertos_españoles_todos)) \
    .select(
        col("aeropuerto_destino_nombre").alias("aeropuerto_nombre"),
        col("aeropuerto_destino_icao").alias("aeropuerto_icao")
    )

df_aeropuertos_extranjeros = df_aeropuertos_origen.unionByName(df_aeropuertos_destino)

df_aeropuertos_extranjeros \
    .groupBy(
        "aeropuerto_nombre",
        "aeropuerto_icao"
    ) \
    .count() \
    .orderBy(col("count").desc()) \
    .display()

Los resultados van avalando todas las conclusiones que hemos sacado previamente. Se realizan muchos más vuelos a nivel continental que a nivel intercontinental como considero que es lógico por la mayor facilidad de realizar un viaje continental frenta a uno intercontinental. Todos los aeropuertos top involucrados son europeos destacando los que ya hemos comentado previamente, los alemanes, Milán, Amsterdam o París.

In [0]:
from pyspark.sql.functions import col, lit, count, sum, when, round

df_aeropuertos_origen = df_vuelos_españa_completo.select(
    col("aeropuerto_origen_nombre").alias("aeropuerto_nombre"),
    col("aeropuerto_origen_icao").alias("aeropuerto_icao"),
    col("viaje_internacional")
)

df_aeropuertos_destino = df_vuelos_españa_completo.select(
    col("aeropuerto_destino_nombre").alias("aeropuerto_nombre"),
    col("aeropuerto_destino_icao").alias("aeropuerto_icao"),
    col("viaje_internacional")
)

df_aeropuertos = df_aeropuertos_origen.unionByName(df_aeropuertos_destino)

df_porcentaje_aeropuertos = df_aeropuertos \
    .groupBy(
        "aeropuerto_nombre",
        "aeropuerto_icao"
    ) \
    .agg(
        count("*").alias("total_viajes"),
        sum(
            when(col("viaje_internacional"), 1).otherwise(0)
        ).alias("viajes_internacionales"),
        sum(
            when(~col("viaje_internacional"), 1).otherwise(0)
        ).alias("viajes_no_internacionales")
    ) \
    .withColumn(
        "porcentaje_internacionales",
        round(
            col("viajes_internacionales") / col("total_viajes") * 100,
            2
        )
    ) \
    .withColumn(
        "porcentaje_no_internacionales",
        round(
            col("viajes_no_internacionales") / col("total_viajes") * 100,
            2
        )
    ) \
    .orderBy(col("total_viajes").desc())

display(df_porcentaje_aeropuertos)

Es muy interesante este análisis podemos ver como los aeropuertos de las islas (Tenerife Norte y Gran Canaria) solo han realizado viajes nacionales.

In [0]:
df_aeropuertos_origen = df_vuelos_españa_completo \
    .filter(col("viaje_corto")) \
    .select(
        col("aeropuerto_origen_nombre").alias("aeropuerto_nombre"),
        col("aeropuerto_origen_icao").alias("aeropuerto_icao"),
        lit("origen").alias("tipo")
    )

df_aeropuertos_destino = df_vuelos_españa_completo \
    .filter(col("viaje_corto")) \
    .select(
        col("aeropuerto_destino_nombre").alias("aeropuerto_nombre"),
        col("aeropuerto_destino_icao").alias("aeropuerto_icao"),
        lit("destino").alias("tipo")
    )

df_aeropuertos = df_aeropuertos_origen.unionByName(df_aeropuertos_destino)

df_aeropuertos \
    .groupBy(
        "aeropuerto_nombre",
        "aeropuerto_icao"
    ) \
    .count() \
    .orderBy(col("count").desc()) \
    .display()

In [0]:
df_aeropuertos_origen = df_vuelos_españa_completo \
    .filter(col("viaje_mediano")) \
    .select(
        col("aeropuerto_origen_nombre").alias("aeropuerto_nombre"),
        col("aeropuerto_origen_icao").alias("aeropuerto_icao"),
        lit("origen").alias("tipo")
    )

df_aeropuertos_destino = df_vuelos_españa_completo \
    .filter(col("viaje_mediano")) \
    .select(
        col("aeropuerto_destino_nombre").alias("aeropuerto_nombre"),
        col("aeropuerto_destino_icao").alias("aeropuerto_icao"),
        lit("destino").alias("tipo")
    )

df_aeropuertos = df_aeropuertos_origen.unionByName(df_aeropuertos_destino)

df_aeropuertos \
    .groupBy(
        "aeropuerto_nombre",
        "aeropuerto_icao"
    ) \
    .count() \
    .orderBy(col("count").desc()) \
    .display()

In [0]:
df_aeropuertos_origen = df_vuelos_españa_completo \
    .filter(col("viaje_largo")) \
    .select(
        col("aeropuerto_origen_nombre").alias("aeropuerto_nombre"),
        col("aeropuerto_origen_icao").alias("aeropuerto_icao"),
        lit("origen").alias("tipo")
    )

df_aeropuertos_destino = df_vuelos_españa_completo \
    .filter(col("viaje_largo")) \
    .select(
        col("aeropuerto_destino_nombre").alias("aeropuerto_nombre"),
        col("aeropuerto_destino_icao").alias("aeropuerto_icao"),
        lit("destino").alias("tipo")
    )

df_aeropuertos = df_aeropuertos_origen.unionByName(df_aeropuertos_destino)

df_aeropuertos \
    .groupBy(
        "aeropuerto_nombre",
        "aeropuerto_icao"
    ) \
    .count() \
    .orderBy(col("count").desc()) \
    .display()

In [0]:
from pyspark.sql.functions import col, lit, count, sum, when, round

df_aeropuertos_origen = df_vuelos_españa_completo.select(
    col("aeropuerto_origen_nombre").alias("aeropuerto_nombre"),
    col("aeropuerto_origen_icao").alias("aeropuerto_icao"),
    col("viaje_corto"),
    col("viaje_mediano"),
    col("viaje_largo")
)

df_aeropuertos_destino = df_vuelos_españa_completo.select(
    col("aeropuerto_destino_nombre").alias("aeropuerto_nombre"),
    col("aeropuerto_destino_icao").alias("aeropuerto_icao"),
    col("viaje_corto"),
    col("viaje_mediano"),
    col("viaje_largo")
)

df_aeropuertos = df_aeropuertos_origen.unionByName(df_aeropuertos_destino)

df_porcentaje_aeropuertos = df_aeropuertos \
    .groupBy(
        "aeropuerto_nombre",
        "aeropuerto_icao"
    ) \
    .agg(
        count("*").alias("total_viajes"),
        sum(
            when(col("viaje_corto"), 1).otherwise(0)
        ).alias("viajes_cortos"),
        sum(
            when(col("viaje_mediano"), 1).otherwise(0)
        ).alias("viajes_medios"),
        sum(
            when(col("viaje_largo"), 1).otherwise(0)
        ).alias("viajes_largos")
    ) \
    .withColumn(
        "porcentaje_cortos",
        round(
            col("viajes_cortos") / col("total_viajes") * 100,
            2
        )
    ) \
    .withColumn(
        "porcentaje_medios",
        round(
            col("viajes_medios") / col("total_viajes") * 100,
            2
        )
    ) \
    .withColumn(
        "porcentaje_largos",
        round(
            col("viajes_largos") / col("total_viajes") * 100,
            2
        )
    ) \
    .orderBy(col("total_viajes").desc())

display(df_porcentaje_aeropuertos)

Está estadística tiene mucho valor porque podemos sacar ciertas conclusiones.

Una conclusión clara y evidente es que la mayoría de los viajes intercontinentales salen o bien de Madrid o bien de Barcelona, es decir los viajes largos salen de esas ciudades normalmente.

Por otro lado, del resto de aeropuertos podemos concluir que la mayoría de los viajes que aparecen son o bien nacionales o bien europeos a ciudades no muy lejanas ya que como podemos observar su porcentaje de viajes es muy superior en viajes cortos y medios a viajes largos.

## Análisis estadísticas relaciondas con las aerolíneas

En este bloque vamos a centrarnos más en los análisis relacionados con las aerolíneas. Una vez hecho esto relacionaremos aerolíneas con aeropuertos, con información específica relativa al avión, rutas y demás. 

In [0]:
df_vuelos_españa_completo.filter(~col("viaje_internacional")).groupBy(col("aerolinea_nombre"),col("aerolinea_icao")).count().orderBy(col("count").desc()).display()

In [0]:
df_vuelos_españa_completo.filter(col("viaje_internacional")).groupBy(col("aerolinea_nombre"),col("aerolinea_icao")).count().orderBy(col("count").desc()).display()

Podemos observar mucha diferencia entre las aerolíneas más centradas en vuelos nacionales como son Vueling, Canair y Naysa y las más centradas en vuelos internacionales como son Ryanair, Iberia o Eurowings Europe. 

In [0]:
df_vuelos_españa_completo.filter(col("viaje_corto")).groupBy(col("aerolinea_nombre"),col("aerolinea_icao")).count().orderBy(col("count").desc()).display()

In [0]:
df_vuelos_españa_completo.filter(col("viaje_mediano")).groupBy(col("aerolinea_nombre"),col("aerolinea_icao")).count().orderBy(col("count").desc()).display()

In [0]:
df_vuelos_españa_completo.filter(col("viaje_largo")).groupBy(col("aerolinea_nombre"),col("aerolinea_icao")).count().orderBy(col("count").desc()).display()

Estos resultados respaldan la conclusión aterior ya que en los viajes cortos destacan los que destacaban en viajes nacionales y a medida que aumenta el tiempo de los viajes van destacando los que destacan en viajes internacionales.

In [0]:
df_vuelos_españa_completo.filter(col("viaje_nocturno")).groupBy(col("aerolinea_nombre"),col("aerolinea_icao")).count().orderBy(col("count").desc()).display()

In [0]:
df_vuelos_españa_completo.filter(col("madrid_or_barcelona")).groupBy(col("aerolinea_nombre"),col("aerolinea_icao")).count().orderBy(col("count").desc()).display()

En vuelos en los que están involucrados Madrid o Barcelona la aerolínea más utilizada es Iberia, esto es bastante coherente ya que de Madrid y Barcelona salen los vuelos más largos normalmente e Iberia es la compañia que más vuelos largos suele llevar a cabo.

Analizamos la relación aeropuerto - aerolínea

In [0]:
df_aeropuertos_origen = df_vuelos_españa_completo \
    .select(
        col("aeropuerto_origen_nombre").alias("aeropuerto_nombre"),
        col("aeropuerto_origen_icao").alias("aeropuerto_icao"),
        col("aerolinea_nombre").alias("aerolinea_nombre"),
        lit("origen").alias("tipo")
    )

df_aeropuertos_destino = df_vuelos_españa_completo \
    .select(
        col("aeropuerto_destino_nombre").alias("aeropuerto_nombre"),
        col("aeropuerto_destino_icao").alias("aeropuerto_icao"),
        col("aerolinea_nombre").alias("aerolinea_nombre"),
        lit("destino").alias("tipo")
    )

df_aeropuertos_aerolineas = df_aeropuertos_origen.unionByName(df_aeropuertos_destino)

df_aeropuertos_aerolineas \
    .groupBy(
        "aeropuerto_nombre",
        "aeropuerto_icao",
        "aerolinea_nombre"
    ) \
    .count() \
    .orderBy(col("count").desc()) \
    .display()

Sacamos por cada aerolínea el aeropuerto donde está involucrada en más viajes

In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window
window_aerolinea = Window.partitionBy("aerolinea_nombre").orderBy(col("count").desc())

df_top_aeropuerto_por_aerolinea = df_aeropuertos_aerolineas \
    .groupBy(
        "aerolinea_nombre",
        "aeropuerto_nombre",
        "aeropuerto_icao"
    ) \
    .count() \
    .withColumn(
        "ranking",
        row_number().over(window_aerolinea)
    ) \
    .filter(col("ranking") == 1) \
    .drop("ranking") \
    .orderBy(col("count").desc())

display(df_top_aeropuerto_por_aerolinea)

Sacamos por cada aeropuerto el módelo de avión que más viajes lleva a cabo en ese aeropuerto

In [0]:
df_aeropuertos_origen = df_vuelos_españa_completo \
    .select(
        col("aeropuerto_origen_nombre").alias("aeropuerto_nombre"),
        col("aeropuerto_origen_icao").alias("aeropuerto_icao"),
        col("modelo_avion").alias("modelo_avion"),
        lit("origen").alias("tipo")
    )

df_aeropuertos_destino = df_vuelos_españa_completo \
    .select(
        col("aeropuerto_destino_nombre").alias("aeropuerto_nombre"),
        col("aeropuerto_destino_icao").alias("aeropuerto_icao"),
        col("modelo_avion").alias("modelo_avion"),
        lit("destino").alias("tipo")
    )

df_aeropuertos_extranjeros = df_aeropuertos_origen.unionByName(df_aeropuertos_destino)

df_aeropuertos_extranjeros \
    .groupBy(
        "aeropuerto_nombre",
        "aeropuerto_icao",
        "modelo_avion"
    ) \
    .count() \
    .orderBy(col("count").desc()) \
    .display()

## Análisis estadísticas relaciondas con las rutas

En primer lugar vamos a sacar las rutas más repetidas respetando origen y destino.

In [0]:
df_vuelos_españa_completo \
    .groupBy(
        col("aeropuerto_origen_nombre"),
        col("aeropuerto_destino_nombre")
    ).count().orderBy(col("count").desc()).display()

Sin duda con una gran difrencia la ruta más repetida es Tenerife Norte - La Palma, y la ruta de vuelta La Palma - Tenerife Norte. Después aparecen muchas conexiones con el aeropuerto de Mallorca siempre como destino como ya habíamos comentado antes muchas de ellas con Alemania como origen. También es una conexión muy repetida La Palma - Gran Canaria.

Lo que nos hace ver la gran conexión y facilidad que existe para moverse en el archipielago canario.

Vamos a analizar las conexiones más repetidas independientemente de quien actua de origen y quien de destino.

In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window
df_rutas = df_vuelos_españa_completo.select(
    when(
        col("aeropuerto_origen_icao") < col("aeropuerto_destino_icao"),
        col("aeropuerto_origen_nombre")
    ).otherwise(col("aeropuerto_destino_nombre")).alias("aeropuerto_1_nombre"),

    when(
        col("aeropuerto_origen_icao") < col("aeropuerto_destino_icao"),
        col("aeropuerto_origen_icao")
    ).otherwise(col("aeropuerto_destino_icao")).alias("aeropuerto_1_icao"),

    when(
        col("aeropuerto_origen_icao") < col("aeropuerto_destino_icao"),
        col("aeropuerto_destino_nombre")
    ).otherwise(col("aeropuerto_origen_nombre")).alias("aeropuerto_2_nombre"),

    when(
        col("aeropuerto_origen_icao") < col("aeropuerto_destino_icao"),
        col("aeropuerto_destino_icao")
    ).otherwise(col("aeropuerto_origen_icao")).alias("aeropuerto_2_icao")
)

df_combinaciones = df_rutas \
    .groupBy(
        "aeropuerto_1_nombre",
        "aeropuerto_1_icao",
        "aeropuerto_2_nombre",
        "aeropuerto_2_icao"
    ) \
    .count() \
    .orderBy(col("count").desc())

df_rutas_por_aeropuerto = df_combinaciones.select(
    col("aeropuerto_1_nombre").alias("aeropuerto_nombre"),
    col("aeropuerto_1_icao").alias("aeropuerto_icao"),
    col("aeropuerto_2_nombre").alias("aeropuerto_pareja_nombre"),
    col("aeropuerto_2_icao").alias("aeropuerto_pareja_icao"),
    col("count")
)

window_aeropuerto = Window.partitionBy("aeropuerto_icao").orderBy(col("count").desc())

df_top_combinacion_por_aeropuerto = df_rutas_por_aeropuerto \
    .withColumn(
        "ranking",
        row_number().over(window_aeropuerto)
    ) \
    .filter(col("ranking") == 1) \
    .drop("ranking") \
    .orderBy(col("count").desc())

display(df_top_combinacion_por_aeropuerto)

Cambia poco respecto a lo ya analizado previamente. Lo más destacado es que a la ya mencionada conexión canaria y a la conexión Alemania - Mallorca se suma la conexión Bilbao - Madrid o Ibiza - Valencia

Sacamos la conexión que más se repite por cada aeropuerto.

In [0]:
df_rutas = df_vuelos_españa_completo.select(
    when(
        col("aeropuerto_origen_icao") < col("aeropuerto_destino_icao"),
        col("aeropuerto_origen_nombre")
    ).otherwise(col("aeropuerto_destino_nombre")).alias("aeropuerto_1_nombre"),

    when(
        col("aeropuerto_origen_icao") < col("aeropuerto_destino_icao"),
        col("aeropuerto_origen_icao")
    ).otherwise(col("aeropuerto_destino_icao")).alias("aeropuerto_1_icao"),

    when(
        col("aeropuerto_origen_icao") < col("aeropuerto_destino_icao"),
        col("aeropuerto_destino_nombre")
    ).otherwise(col("aeropuerto_origen_nombre")).alias("aeropuerto_2_nombre"),

    when(
        col("aeropuerto_origen_icao") < col("aeropuerto_destino_icao"),
        col("aeropuerto_destino_icao")
    ).otherwise(col("aeropuerto_origen_icao")).alias("aeropuerto_2_icao")
)

df_rutas_contadas = df_rutas \
    .groupBy(
        "aeropuerto_1_nombre",
        "aeropuerto_1_icao",
        "aeropuerto_2_nombre",
        "aeropuerto_2_icao"
    ) \
    .count()

df_rutas_aeropuerto_1 = df_rutas_contadas.select(
    col("aeropuerto_1_nombre").alias("aeropuerto_nombre"),
    col("aeropuerto_1_icao").alias("aeropuerto_icao"),
    col("aeropuerto_2_nombre").alias("aeropuerto_pareja_nombre"),
    col("aeropuerto_2_icao").alias("aeropuerto_pareja_icao"),
    col("count").alias("numero_vuelos")
)

df_rutas_aeropuerto_2 = df_rutas_contadas.select(
    col("aeropuerto_2_nombre").alias("aeropuerto_nombre"),
    col("aeropuerto_2_icao").alias("aeropuerto_icao"),
    col("aeropuerto_1_nombre").alias("aeropuerto_pareja_nombre"),
    col("aeropuerto_1_icao").alias("aeropuerto_pareja_icao"),
    col("count").alias("numero_vuelos")
)

df_rutas_por_aeropuerto = df_rutas_aeropuerto_1.unionByName(df_rutas_aeropuerto_2)

window_aeropuerto = Window.partitionBy("aeropuerto_icao") \
    .orderBy(col("numero_vuelos").desc())

df_ruta_mas_frecuente_por_aeropuerto = df_rutas_por_aeropuerto \
    .withColumn(
        "ranking",
        row_number().over(window_aeropuerto)
    ) \
    .filter(col("ranking") == 1) \
    .drop("ranking") \
    .orderBy(col("numero_vuelos").desc())

display(df_ruta_mas_frecuente_por_aeropuerto)

Sacamos la conexión más realizada por cada aerolínea

In [0]:
df_rutas_aerolinea = df_vuelos_españa_completo.select(
    col("aerolinea_nombre"),

    when(
        col("aeropuerto_origen_icao") < col("aeropuerto_destino_icao"),
        col("aeropuerto_origen_nombre")
    ).otherwise(col("aeropuerto_destino_nombre")).alias("aeropuerto_1_nombre"),

    when(
        col("aeropuerto_origen_icao") < col("aeropuerto_destino_icao"),
        col("aeropuerto_origen_icao")
    ).otherwise(col("aeropuerto_destino_icao")).alias("aeropuerto_1_icao"),

    when(
        col("aeropuerto_origen_icao") < col("aeropuerto_destino_icao"),
        col("aeropuerto_destino_nombre")
    ).otherwise(col("aeropuerto_origen_nombre")).alias("aeropuerto_2_nombre"),

    when(
        col("aeropuerto_origen_icao") < col("aeropuerto_destino_icao"),
        col("aeropuerto_destino_icao")
    ).otherwise(col("aeropuerto_origen_icao")).alias("aeropuerto_2_icao")
)

df_rutas_aerolinea_contadas = df_rutas_aerolinea \
    .groupBy(
        "aerolinea_nombre",
        "aeropuerto_1_nombre",
        "aeropuerto_1_icao",
        "aeropuerto_2_nombre",
        "aeropuerto_2_icao"
    ) \
    .count()

window_aerolinea = Window.partitionBy("aerolinea_nombre") \
    .orderBy(col("count").desc())

df_ruta_mas_frecuente_por_aerolinea = df_rutas_aerolinea_contadas \
    .withColumn(
        "ranking",
        row_number().over(window_aerolinea)
    ) \
    .filter(col("ranking") == 1) \
    .drop("ranking") \
    .orderBy(col("count").desc())

display(df_ruta_mas_frecuente_por_aerolinea)

## Análisis estadísticas relaciondas con el avión

Sacamos una serie de estadísticas relacionadas con el modelo del avión, matrícula y asociandolas a aerolíneas y aeropuertos

In [0]:
df_vuelos_españa_completo.groupBy(col("modelo_avion"), col("aerolinea_nombre")).count().orderBy(col("count").desc()).display()

In [0]:
df_vuelos_españa_completo.groupBy(col("matricula_avion"), col("aerolinea_nombre")).count().orderBy(col("count").desc()).display()

Sacamos lás matrículas de avión que más viajes han realizado

In [0]:
df_vuelos_españa_completo.groupBy(col("matricula_avion")).count().orderBy(col("count").desc()).display()

In [0]:
df_vuelos_españa_completo.filter(~col("viaje_internacional")).groupBy(col("modelo_avion"), col("aerolinea_nombre")).count().orderBy(col("count").desc()).display()

In [0]:
df_vuelos_españa_completo.filter(col("viaje_internacional")).groupBy(col("modelo_avion"), col("aerolinea_nombre")).count().orderBy(col("count").desc()).display()

In [0]:
df_vuelos_españa_completo.filter(~col("viaje_internacional")).groupBy(col("matricula_avion"), col("aerolinea_nombre")).count().orderBy(col("count").desc()).display()

In [0]:
df_vuelos_españa_completo.filter(col("viaje_internacional")).groupBy(col("matricula_avion"), col("aerolinea_nombre")).count().orderBy(col("count").desc()).display()

In [0]:
df_vuelos_españa_completo.filter(col("viaje_corto")).groupBy(col("modelo_avion"), col("aerolinea_nombre")).count().orderBy(col("count").desc()).display()

In [0]:
df_vuelos_españa_completo.filter(col("viaje_mediano")).groupBy(col("modelo_avion"), col("aerolinea_nombre")).count().orderBy(col("count").desc()).display()

In [0]:
df_vuelos_españa_completo.filter(col("viaje_largo")).groupBy(col("modelo_avion"), col("aerolinea_nombre")).count().orderBy(col("count").desc()).display()

In [0]:
df_vuelos_españa_completo.filter(col("viaje_corto")).groupBy(col("matricula_avion"), col("aerolinea_nombre")).count().orderBy(col("count").desc()).display()

In [0]:
df_vuelos_españa_completo.filter(col("viaje_mediano")).groupBy(col("matricula_avion"), col("aerolinea_nombre")).count().orderBy(col("count").desc()).display()

In [0]:
df_vuelos_españa_completo.filter(col("viaje_largo")).groupBy(col("matricula_avion"), col("aerolinea_nombre")).count().orderBy(col("count").desc()).display()

## 6. Structured Streaming

Analizamos en tiempo real la entrada de nuevos datos

In [0]:
schema_vuelos = df_vuelos_españa.schema

In [0]:
df_aerolineas_join = df_vuelos_españa_aerolineas.select(
    col("icao").alias("aerolinea_icao_join"),
    col("iata").alias("aerolinea_iata"),
    col("name").alias("aerolinea_nombre")
)

df_aeropuertos_origen = df_vuelos_españa_aeropuertos.select(
    col("icao").alias("origen_icao_join"),
    col("iata").alias("aeropuerto_origen_iata"),
    col("name").alias("aeropuerto_origen_nombre")
)

df_aeropuertos_destino = df_vuelos_españa_aeropuertos.select(
    col("icao").alias("destino_icao_join"),
    col("iata").alias("aeropuerto_destino_iata"),
    col("name").alias("aeropuerto_destino_nombre")
)

In [0]:
df_vuelos_streaming = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .option("maxFilesPerTrigger", 1) \
    .schema(schema_vuelos) \
    .load("/Volumes/vuelos_españa/default/vuelos_streaming_entrada/")

In [0]:
df_vuelos_streaming_completo = (
    df_vuelos_streaming
    .join(
        df_aerolineas_join,
        col("aerolinea_icao") == col("aerolinea_icao_join"),
        "left"
    )
    .join(
        df_aeropuertos_origen,
        col("aeropuerto_origen_icao") == col("origen_icao_join"),
        "left"
    )
    .join(
        df_aeropuertos_destino,
        col("aeropuerto_destino_icao") == col("destino_icao_join"),
        "left"
    )
    .drop(
        "aerolinea_icao_join",
        "origen_icao_join",
        "destino_icao_join"
    )
)

In [0]:
df_vuelos_streaming_completo = df_vuelos_streaming_completo.select(
    "id",
    "aeropuerto_origen_nombre",
    "aeropuerto_origen_icao",
    "aeropuerto_origen_iata",
    "aeropuerto_destino_nombre",
    "aeropuerto_destino_icao",
    "aeropuerto_destino_iata",
    "aerolinea_nombre",
    "aerolinea_icao",
    "aerolinea_iata",
    "fecha_hora_despegue",
    "fecha_hora_aterrizaje",
    "numero_vuelo",
    "matricula_avion",
    "modelo_avion",
    "indicativo"
)

In [0]:
df_kpi_vuelos_por_aerolinea = df_vuelos_streaming_completo \
    .groupBy("aerolinea_nombre") \
    .count()

In [0]:
df_kpi_vuelos_por_origen = df_vuelos_streaming_completo \
    .groupBy(
        "aeropuerto_origen_nombre",
        "aeropuerto_origen_icao"
    ) \
    .count()

In [0]:
df_kpi_vuelos_por_destino = df_vuelos_streaming_completo \
    .groupBy(
        "aeropuerto_destino_nombre",
        "aeropuerto_destino_icao"
    ) \
    .count()

In [0]:
ruta_checkpoint_base = "/Volumes/vuelos_españa/default/checkpoints_streaming/"

In [0]:
query_vuelos_por_aerolinea = df_kpi_vuelos_por_aerolinea.writeStream \
    .format("memory") \
    .queryName("kpi_vuelos_por_aerolinea") \
    .outputMode("complete") \
    .option("checkpointLocation", ruta_checkpoint_base + "kpi_vuelos_por_aerolinea") \
    .trigger(availableNow=True) \
    .start()

query_vuelos_por_origen = df_kpi_vuelos_por_origen.writeStream \
    .format("memory") \
    .queryName("kpi_vuelos_por_origen") \
    .outputMode("complete") \
    .option("checkpointLocation", ruta_checkpoint_base + "kpi_vuelos_por_origen") \
    .trigger(availableNow=True) \
    .start()

query_vuelos_por_destino = df_kpi_vuelos_por_destino.writeStream \
    .format("memory") \
    .queryName("kpi_vuelos_por_destino") \
    .outputMode("complete") \
    .option("checkpointLocation", ruta_checkpoint_base + "kpi_vuelos_por_destino") \
    .trigger(availableNow=True) \
    .start()

In [0]:
spark.table("kpi_vuelos_por_aerolinea") \
    .orderBy(col("count").desc()) \
    .display()

In [0]:
spark.table("kpi_vuelos_por_origen") \
    .orderBy(col("count").desc()) \
    .display()

In [0]:
spark.table("kpi_vuelos_por_destino") \
    .orderBy(col("count").desc()) \
    .display()

In [0]:
for stream in spark.streams.active:
    stream.stop()


## 7. Conclusiones y limitaciones

En este proyecto se ha realizado un análisis de vuelos relacionados con aeropuertos españoles utilizando Apache Spark. A partir de los datos originales obtenidos mediante FlightRadar24, se cargaron ficheros CSV con información básica de vuelos y posteriormente se enriquecieron mediante joins con datasets auxiliares de aeropuertos y aerolíneas, incorporando nombres, códigos IATA e ICAO y otra información descriptiva.

Durante el desarrollo del notebook se revisó la estructura del dataset, se analizaron valores nulos y se crearon nuevas variables para facilitar el análisis, como la duración del vuelo, el tipo de trayecto según su duración, la identificación de viajes internacionales y otras columnas derivadas de las fechas de despegue y aterrizaje. Gracias a estas transformaciones se pudieron estudiar patrones relacionados con aeropuertos más utilizados, aerolíneas con mayor presencia, rutas más frecuentes y distribución de vuelos según origen y destino.

Entre las conclusiones principales, se observa que algunos aeropuertos concentran gran parte del tráfico analizado, actuando como puntos principales de conexión. También se aprecia que determinadas aerolíneas tienen una presencia mucho mayor en el conjunto de vuelos, y que existen rutas que se repiten con bastante frecuencia dentro del periodo estudiado. Además, el análisis permite diferenciar entre vuelos nacionales e internacionales y estudiar el comportamiento de los trayectos según su duración.

Como parte final del proyecto, se ha incluido una aproximación a Structured Streaming para simular la llegada progresiva de nuevos ficheros CSV. Aunque los datos utilizados no proceden de una fuente en tiempo real real, esta sección permite poner en práctica el procesamiento incremental de datos y la actualización de indicadores a medida que se incorporan nuevos vuelos.

Como limitaciones, el análisis depende de los datos disponibles en los ficheros descargados, por lo que puede haber vuelos incompletos, valores nulos o información no disponible para determinados aeropuertos o aerolíneas. Además, el periodo temporal analizado es limitado, por lo que los resultados no deben interpretarse como una representación completa del tráfico aéreo español. Por último, la parte de streaming se plantea como una simulación basada en archivos CSV, y no como una conexión directa a una fuente de datos en tiempo real como una API o Kafka.
